<a href="https://colab.research.google.com/github/ms24191-PN/IDC-AQIupgrade/blob/main/Complete_data_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading the data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rohanrao/air-quality-data-in-india")

print("Path to dataset files:", path)

import os
print(os.listdir(path))

In [ ]:
import pandas as pd

file_path = os.path.join(path, 'city_day.csv')
data = pd.read_csv(file_path)
data = data.dropna(subset=['AQI']) #Deleting the null values
#print (data_final.head())

print('DATA.SHAPE : ',"\n", data.shape, "\n", 'DATA.COLUMNS : ', "\n", data.columns, "\n",'DATA.HEAD : ', "\n",  data.head, "\n",'DATA.INFO', "\n",  data.info, "\n",'DATA.DESCRIBE',"\n", data.describe)

# Filling NaN values of Pollutant

In [ ]:
pollutants = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'
]

# Replace missing values with the median of each pollutant
data[pollutants] = data[pollutants].fillna(data[pollutants].median())

# Check missing values after imputation
print(data[pollutants].isnull().sum())


# Data Preprocessing

In [ ]:
print('NULL VALUES FOR RESPECTIVE COLUMNS :','\n', data.isnull().sum(),'\n', data.duplicated().sum())
print('DUPLICATED VALUES IN THE DATA :','\n',data.duplicated().sum()) #Checking the duplicate values
print('NO OF UNIQUE CITIES : ','\n',data['City'].unique())  #Checking the inconsistent city/station names.
print('NO VALUES PER CITIES : ', '\n', data['City'].value_counts())
print('DTYPE OD DATE : ', '\n',data['Date'].dtype)

## Data feature addition

### Defining the Season function

In [ ]:
def get_season(month):
  #Standard seson month in NH are used here
  if month in [12,1,2]:
    return "Winter"
  elif month in [3,4,5]:
    return "Spring"
  elif month in [6,7,8]:
    return "Summer"
  else:
    return "Monsoon"

In [ ]:
data['Date'] = pd.to_datetime(data['Date'])   #COnverting the dtype from object to datetime
data['Year'] = data['Date'].dt.year
data['Month']= data['Date'].dt.month     #Creating month data feature
data['Weekday'] = data['Date'].dt.day_name()   #Creating weekday data feature
data['Day'] = data['Date'].dt.day
data['Season'] = data['Month'].apply(get_season)     #Creating season  data feature
data.head()
print(data.groupby('Season')['AQI'].mean())
data['Winter'] = data['Month'].isin([12,1,2]).astype(int) #Here 1 -> Winter month and 0 -> other month
print(data.groupby('Winter')['AQI'].mean())      #Creating Winter indicator data feature

Downloading the Feature and Target data as csv

In [ ]:
#from google.colab import files

#data.to_csv('AQI_data.csv', index=False)
#files.download('AQI_data.csv')

# Visualize city-wise AQI

## City-wise AQI

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

#let calculate the average of the year
avg_year_aqi = (data.groupby(['City','Year'])['AQI'].mean().reset_index())

cities = sorted(avg_year_aqi['City'].unique())
colors = [
    "#E6194B", "#3CB44B", "#4363D8", "#F58231",
    "#911EB4", "#42D4F4", "#F032E6", "#BFEF45",
    "#FABED4", "#469990", "#DCBEFF", "#9A6324",
    "#800000", "#AAFFC3", "#000075", "#A9A9A9",
    "#FF0000", "#00FF00", "#0000FF", "#FFD700",
    "#FF1493", "#00CED1", "#FF8C00", "#4B0082",
    "#8B4513", "#2F4F4F"
]
plt.figure(figsize = (16,8))

for city , color in zip(cities, colors):
  city_data = avg_year_aqi[avg_year_aqi['City'] == city].sort_values('Year')
  plt.plot(city_data['Year'], city_data['AQI'], marker = 'o', linewidth =2, markersize = 5, label= city, color = color)

plt.title('Annual aqi of the city-wise', fontsize = 16)
plt.xlabel('Year')
plt.ylabel('Average AQI')

plt.xticks(sorted(avg_year_aqi['Year'].unique()), rotation = 45)
plt.legend(title = 'City', bbox_to_anchor = (1.02,1), loc ='upper left')
plt.grid(True, alpha = 0.3)
plt.tight_layout()
plt.show()

For top 10 Cities


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Count available AQI data points for each city
city_counts = (data.dropna(subset=['AQI']).groupby('City')['AQI'].count().sort_values(ascending=False))

#Select top 10 cities with most AQI data points
top_10_cities = city_counts.head(10).index.tolist()

#Filtering data for selected cities
top_data = data[data['City'].isin(top_10_cities)].copy()

#Calculate annual average AQI for each city
yearly_avg_aqi = (top_data.groupby(['City', 'Year'])['AQI'].mean().reset_index())

#Plot all 10 cities in a single graph
colors = sns.color_palette('husl', n_colors=len(top_10_cities))

plt.figure(figsize=(16, 8))

for city, color in zip(top_10_cities, colors):

    city_data = yearly_avg_aqi[yearly_avg_aqi['City'] == city].sort_values('Year')

    plt.plot(
        city_data['Year'],
        city_data['AQI'],
        marker='o',
        linewidth=2,
        markersize=5,
        label=city,
        color=color
    )

plt.title('Annual Average AQI: Top 10 Cities by Data Availability',fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Annual Average AQI', fontsize=12)

plt.xticks(sorted(yearly_avg_aqi['Year'].unique()),rotation=45)
plt.legend(title='City',bbox_to_anchor=(1.02, 1),loc='upper left')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# 4. Calculate overall average AQI for each selected city
city_avg_aqi = (top_data.groupby('City').agg(Data_Points=('AQI', 'count'),Average_AQI=('AQI', 'mean')).reset_index())

# Rank cities by average AQI
city_avg_aqi = city_avg_aqi.sort_values('Average_AQI',ascending=False).reset_index(drop=True)

city_avg_aqi['Average_AQI'] = city_avg_aqi['Average_AQI'].round(2)

# Display the complete ranking table
print("AQI Ranking of the Top 10 Cities")
display(city_avg_aqi)

# 5. Identify the highest and lowest average AQI cities
highest_aqi_city = city_avg_aqi.loc[city_avg_aqi['Average_AQI'].idxmax()]

lowest_aqi_city = city_avg_aqi.loc[city_avg_aqi['Average_AQI'].idxmin()]

# Create summary table
summary_table = pd.DataFrame([
    {
        'Category': 'Highest Average AQI',
        'City': highest_aqi_city['City'],
        'Average AQI': highest_aqi_city['Average_AQI'],
        'Data Points': highest_aqi_city['Data_Points']
    },
    {
        'Category': 'Lowest Average AQI',
        'City': lowest_aqi_city['City'],
        'Average AQI': lowest_aqi_city['Average_AQI'],
        'Data Points': lowest_aqi_city['Data_Points']
    }
])

print("Cities with the Highest and Lowest Average AQI")
display(summary_table)

For Mumbai , Bengaluru, and Delhi

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


# Filter dates and cities
cities = ['Delhi', 'Bengaluru', 'Chennai',
          'Hyderabad', 'Mumbai', 'Chandigarh']

plot_data = data[data['City'].isin(cities) & data['Date'].between('2019-09-03', '2020-07-01')].dropna(subset=['Date', 'AQI']).copy()

plot_data = plot_data.sort_values('Date')

# Plot all available data points
plt.figure(figsize=(16, 8))

for city in cities:
    city_data = plot_data[plot_data['City'] == city]

    plt.plot(
        city_data['Date'],
        city_data['AQI'],
        marker='o',
        markersize=3,
        linewidth=1.5,
        label=city
    )

plt.title('AQI Trends Across Six Cities (Sep 2019 – Jul 2020)', fontsize=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('AQI', fontsize=12)
plt.legend(title='City')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


# Define cities and date range
cities = ['Delhi', 'Bengaluru', 'Chennai',
    'Hyderabad', 'Mumbai', 'Chandigarh']

start_date = pd.Timestamp('2019-09-03')
end_date = pd.Timestamp('2020-07-01')

# Filter data
plot_data = data[data['City'].isin(cities) & data['Date'].between(start_date, end_date)].dropna(subset=['Date', 'AQI']).copy()

# Create Year-Month column and calculate monthly average AQI
plot_data['Month'] = plot_data['Date'].dt.to_period('M')

monthly_aqi = (plot_data.groupby(['City', 'Month'])['AQI'].mean().reset_index())

# Converting Month to datetime for plotting the data
monthly_aqi['Month'] = monthly_aqi['Month'].dt.to_timestamp()

# Defining the colors
colors = {
    'Delhi': '#E6194B',
    'Bengaluru': '#3CB44B',
    'Chennai': '#4363D8',
    'Hyderabad': '#F58231',
    'Mumbai': '#911EB4',
    'Chandigarh': '#469990'
}

# Plot monthly average AQI
plt.figure(figsize=(15, 7))

for city in cities:
    city_data = monthly_aqi[monthly_aqi['City'] == city].sort_values('Month')

    plt.plot(
        city_data['Month'],
        city_data['AQI'],
        marker='o',
        linewidth=2,
        markersize=5,
        label=city,
        color=colors[city]
    )

plt.title('Monthly Average AQI for Six Cities ''(Sep 2019 – Jul 2020)',fontsize=15)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Monthly Average AQI', fontsize=12)

plt.xticks(
    monthly_aqi['Month'].drop_duplicates().sort_values(),
    monthly_aqi['Month'].drop_duplicates().sort_values().dt.strftime('%b %Y'),
    rotation=45
)

plt.legend(title='City')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Pollutant correlations

Pollutant

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select pollutant features
pollutants = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'
]

# Create box plot
plt.figure(figsize=(14, 7))

sns.boxplot(data=data[pollutants])

plt.title('Box Plot of Pollutant Features', fontsize=16)
plt.xlabel('Pollutant', fontsize=12)
plt.ylabel('Concentration', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pollutants = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'
]


pollutant_data = data[pollutants].copy()


correlation_matrix = pollutant_data.corr(method='pearson')

# Plotting the correlation heatmap
plt.figure(figsize=(13, 10))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5,
    vmin=-1,
    vmax=1,
    center=0,
    square=True
)

plt.title('Correlation Analysis Among Air Pollutants', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Yearly / Monthly Trends

Without averaging the month

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

cities = sorted(data['City'].unique())
years = sorted(data['Year'].unique())


year_colors = dict(
    zip(years, sns.color_palette('husl', n_colors=len(years)))
)

# Use a leap year - so that February 29th is valid
reference_year = 2020

# Create a common date using the actual month and day
data['Plot_Date'] = pd.to_datetime(
    str(reference_year) + '-' +
    data['Month'].astype(str).str.zfill(2) + '-' +
    data['Day'].astype(str).str.zfill(2)
)

for city in cities:

    # Select the city
    city_data = data[data['City'] == city].copy()

    plt.figure(figsize=(16, 8))

    for year in years:

        # Selecting the year
        year_data = city_data[city_data['Year'] == year].copy()

        if len(year_data) == 0:
            continue

        # Sort from January to December
        year_data = year_data.sort_values('Plot_Date')

        # Ploting actual daily AQI
        plt.plot(
            year_data['Plot_Date'],
            year_data['AQI'],
            label=str(year),
            color=year_colors[year],
            linewidth=1
        )

    plt.title(f'Daily AQI Across Years - {city}')
    plt.xlabel('Month')
    plt.ylabel('AQI')

    # January to December ticks for uniform date
    month_positions = pd.date_range(start=f'{reference_year}-01-01',end=f'{reference_year}-12-01',freq='MS')

    month_labels = ['January', 'February', 'March', 'April',
        'May', 'June', 'July', 'August',
        'September', 'October', 'November', 'December']

    plt.xticks(month_positions,month_labels,rotation=45)

    # Start exactly at January and end at December
    plt.xlim(pd.Timestamp(f'{reference_year}-01-01'),pd.Timestamp(f'{reference_year}-12-31'))

    plt.legend(title='Year',bbox_to_anchor=(1.02, 1),loc='upper left')

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

With averging the month

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create month from the original Date column
#data['Month'] = data['Date'].dt.month

# Calculating monthly AQI for each city and year
monthly_aqi = (data.groupby(['City', 'Year', 'Month'])['AQI'].mean().reset_index())

# Getting cities and years
cities = sorted(monthly_aqi['City'].unique())
years = sorted(monthly_aqi['Year'].unique())

# Creating a different color for each year
year_colors = dict(zip(years, sns.color_palette('husl', n_colors=len(years))))

# Month names
month_labels = ['January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December']

# Ploting one graph for each cities
for city in cities:

    city_data = monthly_aqi[monthly_aqi['City'] == city].copy()

    plt.figure(figsize=(14, 7))

    for year in years:

        year_data = city_data[city_data['Year'] == year].sort_values('Month')

        if len(year_data) == 0:
            continue

        plt.plot(year_data['Month'],year_data['AQI'],marker='o',linewidth=2,label=str(year),color=year_colors[year])

    plt.title(f'Monthly AQI Across Years - {city}')
    plt.xlabel('Month')
    plt.ylabel('AQI')

    plt.xticks(range(1, 13),month_labels,rotation=45)

    plt.xlim(1, 12)

    plt.legend(title='Year',bbox_to_anchor=(1.02, 1),loc='upper left')

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
print(data.columns)

# Train - Test Spilt

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

#Pollutant features
pollutants = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3','CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']


# Preparing features and target
X = data[pollutants].copy()
y = pd.to_numeric(data['AQI'], errors='coerce')


X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)


print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nInput features:")
print(X_train.columns.tolist())

# Model


## Linear Regression Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
y_pred = model_lr.predict(X_test)


print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

Feature Importance

In [ ]:
import pandas as pd

importance_lr = pd.Series(
    abs(model_lr.coef_),
    index=X_train.columns
).sort_values(ascending=False)

print(importance_lr)

## Decision Tree Model

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


model_dt = DecisionTreeRegressor(max_depth=10, min_samples_leaf=20, min_samples_split=2)
model_dt.fit(X_train, y_train)
y_pred = model_dt.predict(X_test)


print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

Feature Importance

In [ ]:
importance_dt = pd.Series(
    model_dt.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance_dt)

## Random Forest Model

In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


model_rf = RandomForestRegressor(n_estimators= 120, min_samples_split= 10, min_samples_leaf= 1, max_features= 0.5, max_depth= 20, bootstrap= True)
model_rf.fit(X_train, y_train)
y_pred = model_rf.predict(X_test)


print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))


Feature importance

In [ ]:
importance = pd.Series(
    model_rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance)

# Next Day Predication Model

Model Regression metrics:


In [ ]:

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)



df = data.copy()

print("Available cities:", df["City"].dropna().unique())

city_name = input("Enter city name: ").strip()

# 2.Finding the city in the list of cities
city_df = df[
    df["City"].str.casefold() == city_name.casefold()
].copy()

if city_df.empty:
    print("City not found. Please check the city name.")

else:

    city_df = city_df.dropna(subset=["Date", "AQI"])

    #Calculating the daily average AQI
    city_df = (
        city_df.groupby("Date", as_index=False)["AQI"]
        .mean()
        .sort_values("Date")
        .reset_index(drop=True)
    )

    #Creating time - series(Single day) feature for model training
    city_df["AQI_lag_1"] = city_df["AQI"].shift(1)
    city_df["AQI_lag_2"] = city_df["AQI"].shift(2)
    city_df["AQI_lag_3"] = city_df["AQI"].shift(3)
    city_df["AQI_lag_7"] = city_df["AQI"].shift(7)

    #Create rolling averages using previous days data
    city_df["AQI_rolling_mean_3"] = (
        city_df["AQI"].shift(1).rolling(window=3).mean()
    )

    city_df["AQI_rolling_mean_7"] = (
        city_df["AQI"].shift(1).rolling(window=7).mean()
    )

    features = ["AQI_lag_1","AQI_lag_2","AQI_lag_3","AQI_lag_7","AQI_rolling_mean_3","AQI_rolling_mean_7"]

    city_df["Target_AQI"] = city_df["AQI"]

    #model_df = city_df.dropna(subset=features + ["Target_AQI"]).reset_index(drop=True)
    #30 just a random number, So that the model can learn the pattern
    if len(model_df) < 30:
        print("Not enough historical data for reliable tuning.")

    else:
        X = model_df[features]
        y = model_df["Target_AQI"]
        X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)


        final_model = RandomForestRegressor(bootstrap = True, max_depth = 15, max_features = 'log2', min_samples_leaf = 8, min_samples_split = 8, n_estimators = 100)
        final_model.fit(X, y)

        #Predicting next day's AQI using the latest observations
        latest_date = city_df.iloc[-1]["Date"]
        latest_aqi = city_df.iloc[-1]["AQI"]

        #using this feature for next day predication
        latest_features = pd.DataFrame([{
            "AQI_lag_1": city_df["AQI"].iloc[-1],
            "AQI_lag_2": city_df["AQI"].iloc[-2],
            "AQI_lag_3": city_df["AQI"].iloc[-3],
            "AQI_lag_7": city_df["AQI"].iloc[-7],
            "AQI_rolling_mean_3": city_df["AQI"].iloc[-3:].mean(),
            "AQI_rolling_mean_7": city_df["AQI"].iloc[-7:].mean()
        }])[features]

        next_prediction = final_model.predict(latest_features)[0]

        print("\nNext-day prediction:")
        print("Latest recorded date:", latest_date.date())
        print("Latest recorded AQI:", round(latest_aqi, 2))
        print("Predicted date:",(latest_date + pd.Timedelta(days=1)).date())
        print( "Predicted next-day AQI:",round(next_prediction, 2))

In [ ]:

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

df = data.copy()

print("Available cities:", df["City"].dropna().unique())
city_name = input("Enter city name: ").strip()

# Find the selected city
city_df = df[
    df["City"].str.casefold() == city_name.casefold()
].copy()

if city_df.empty:
    print("City not found. Please check the city name.")

else:
    city_df["Date"] = pd.to_datetime(
        city_df["Date"], errors="coerce"
    )
    city_df["AQI"] = pd.to_numeric(
        city_df["AQI"], errors="coerce"
    )

    city_df = city_df.dropna(subset=["Date", "AQI"])

    # Calculate daily average AQI
    city_df = (
        city_df.groupby("Date", as_index=False)["AQI"]
        .mean()
        .sort_values("Date")
        .reset_index(drop=True)
    )

    # Create lag features
    city_df["AQI_lag_1"] = city_df["AQI"].shift(1)
    city_df["AQI_lag_2"] = city_df["AQI"].shift(2)
    city_df["AQI_lag_3"] = city_df["AQI"].shift(3)
    city_df["AQI_lag_7"] = city_df["AQI"].shift(7)

    # Create rolling averages using previous observations
    city_df["AQI_rolling_mean_3"] = (
        city_df["AQI"].shift(1).rolling(window=3).mean()
    )

    city_df["AQI_rolling_mean_7"] = (
        city_df["AQI"].shift(1).rolling(window=7).mean()
    )

    features = [
        "AQI_lag_1",
        "AQI_lag_2",
        "AQI_lag_3",
        "AQI_lag_7",
        "AQI_rolling_mean_3",
        "AQI_rolling_mean_7"
    ]

    city_df["Target_AQI"] = city_df["AQI"]

    # Remove rows with missing features
    model_df = city_df.dropna(
        subset=features + ["Target_AQI"]
    ).reset_index(drop=True)

    if len(model_df) < 30:
        print("Not enough historical data for reliable evaluation.")

    else:
        X = model_df[features]
        y = model_df["Target_AQI"]

        # Chronological 80:20 train-test split
        split_index = int(len(model_df) * 0.80)

        X_train = X.iloc[:split_index]
        X_test = X.iloc[split_index:]

        y_train = y.iloc[:split_index]
        y_test = y.iloc[split_index:]

        # Create Random Forest model
        model = RandomForestRegressor(bootstrap = True, max_depth = 15, max_features = 'log2', min_samples_leaf = 8, min_samples_split = 8, n_estimators = 100)

        # Train only on training data
        model.fit(X_train, y_train)

        # Predict AQI for the test period
        predictions = model.predict(X_test)

        # Calculate regression metrics
        mae = mean_absolute_error(y_test, predictions)
        mse = mean_squared_error(y_test, predictions)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, predictions)

        print("\nRegression Metrics for", city_name)
        print("--------------------------------")
        print("Training samples:", len(X_train))
        print("Testing samples:", len(X_test))
        print("MAE:", round(mae, 3))
        print("MSE:", round(mse, 3))
        print("RMSE:", round(rmse, 3))
        print("R2 Score:", round(r2, 3))

        # Compare actual and predicted AQI
        results = pd.DataFrame({
            "Date": model_df["Date"].iloc[split_index:].values,
            "Actual_AQI": y_test.values,
            "Predicted_AQI": predictions
        })

        print("\nActual vs Predicted AQI:")
        print(results.head(10))
